# Analysis for xchem data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm

In [ ]:
from tools import (
    compute_uniqueness,
    compute_novelty,
    compute_unique_novelty,
)

## Load data

In [ ]:
df_cond = pd.read_csv("predictions/conditional_xchem/test_conditional.csv")
df_uncond = pd.read_csv("predictions/conditional_xchem/test_unconditional.csv")

df = pd.concat([df_uncond, df_cond], axis=1, ignore_index=False)

In [ ]:
file = "data/conditional_xchem/mpro_active_site_fragments_separate.csv"
df_cond = pd.read_csv(file)
df_cond = df_cond[df_cond.fail != 1.0]
print(len(df_cond))
reference_smiles = set(df_cond["smiles"])
print(len(reference_smiles))

## Enrich data

In [ ]:
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

In [ ]:
# validity, novely, uniqueness
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]
df["valid_smiles"] = df["valid"].astype(bool) * df["smiles"]
# df["valid_scaffold_rdkit_csk"] = df["valid"].astype(bool) & df[
#     "scaffold_rdkit_csk"
# ].astype(bool)
# df["valid_scaffold_hop_smiles"] = (~df["valid_scaffold_rdkit_csk"]) * df["valid_smiles"]

In [ ]:
# define order of methods
df["method"] = "debug"
order = [
    "debug",
]
df["method"] = pd.Categorical(df["method"], categories=order, ordered=True)

## Tables

### Validity

In [ ]:
n = 45


def mean(x):
    if len(x) == 0:
        return float("nan")
    return np.sum(x) / n


aggs = {
    "total_number": ("total_number", "sum"),
    "generated": ("total_number", mean),
    "connected": ("connected", mean),
    "chemical": ("chemical", mean),
    "physical": ("physical", mean),
    "valid": ("valid", mean),
    # "valid_scaffold_hop": ("scaffold_rdkit_csk", 1 - mean),
}
df_agg = df.groupby(["method"], observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])

### Properties

In [ ]:
df_filter = df[df.valid]
aggs = {
    "qed": ["mean", "std"],
    "sa": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
    "weight": ["mean", "std"],
    "num_heavy": ["mean", "std"],
    "num_rings": ["mean", "std"],
    "lipinski": ["mean", "std"],
    "logp": ["mean", "std"],
    "spacial": ["mean", "std"],
}
df_agg = df_filter.groupby(["method"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Energy ratio

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "ensemble_avg_energy": ["mean", "std"],
    "mol_pred_energy": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
}
df_agg = df_filter.groupby(["method"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Unique, novel, useful

In [ ]:
n = 100000

df_filter = df
aggs = {
    "Valid": ("valid", lambda x: sum(x) / n if len(x) > 2000 else sum(x) / 1127),
    # "Valid & Scaffold Hop": ("scaffold_rdkit_csk", lambda x: 1 - mean(x)),
    # "Valid & Unique": (
    #     "valid_smiles",
    #     lambda x: compute_uniqueness(x, total=1) / n,
    # ),
    # "Valid & Novel": (
    #     "valid_smiles",
    #     lambda x: compute_novelty(x, reference_smiles, total=1) / n,
    # ),
    "Valid & Unique & Novel": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n
        if len(x) > 2000
        else compute_unique_novelty(x, reference_smiles, total=1) / 1127,
    ),
    "Valid & Unique & Novel & Scaffold Hop": (
        "valid_scaffold_hop_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n
        if len(x) > 2000
        else compute_unique_novelty(x, reference_smiles, total=1) / 1127,
    ),
}
df_agg = df_filter.groupby(["method"], observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

### Quality

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "sucos_frag mean": ("sucos_frag", "mean"),
    "sucos_frag std": ("sucos_frag", "std"),
    "sucos_frag > 0.8 ": ("sucos_frag", lambda x: sum(x > 0.8) / n),
    "sucos_link mean": ("sucos_link", "mean"),
    "sucos_link std": ("sucos_link", "std"),
    "sucos_link > 0.55 ": ("sucos_link", lambda x: sum(x > 0.55) / n),
}
df_agg = df_filter.groupby(["method"], observed=False).agg(**aggs)
# df_agg["tanimoto > 0.8 and sucos > 0.55"] = (
#     df_agg["sucos > 0.55 "] * df_agg["tanimoto > 0.8 "]
# )
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

### Usefulness

In [ ]:
# How many unique new linkers capitulate the fragments and have SuCOS larger than x?

# How many good new linkers were created?
threshold_link = 0.7
threshold_frag = 0.7
aggs = {
    f"sucos_frag > {threshold_frag}": (
        "sucos_frag",
        lambda x: sum(x > threshold_frag) > 0,
    ),
    f"sucos_link > {threshold_link}": (
        "sucos_link",
        lambda x: sum(x > threshold_link) > 0,
    ),
}
df_agg = df_best.groupby(["method", "reference_molecule"], observed=True).agg(**aggs)
df_agg[f"sucos_link > {threshold_link} and sucos_frag > {threshold_frag}"] = (
    df_agg[f"sucos_link > {threshold_link}"] * df_agg[f"sucos_frag > {threshold_frag}"]
)
print("number of reference molecules new linkers were created for")
df_agg.groupby(["method"], observed=True).sum()
# df_agg

In [ ]:
sns.histplot(
    df[
        [
            c
            for c in df.columns
            if "sucos" in c
            and not (c.endswith("count") or c.endswith("mean") or c.endswith("std"))
        ]
    ],
    # bins=50,
    kde=True,
    # fill=False,
    element="step",
    # element="poly"
)